# Proyek Forecasting Inflasi Indonesia
### Dari data mentah BPS hingga prediksi 12 bulan yang layak dipresentasikan

Notebook ini merangkum keseluruhan alur proyek: penggabungan data inflasi bulanan BPS (2010–2025), pembangunan model Prophet, evaluasi akurasi, optimasi, dan kesimpulan akhir mengenai pemilihan target prediksi yang tepat.

**Temuan utama (executive summary):**

1. Model forecasting terbaik untuk **presisi angka** menggunakan target **Y-o-Y (year-on-year)**, bukan M-to-M (month-to-month).
2. Membandingkan angka MAE M-to-M vs Y-o-Y secara mentah adalah **"jebakan skala"** — angka error yang lebih besar pada Y-o-Y sebenarnya berarti model yang **lebih akurat** secara relatif.
3. Estimasi inflasi tahunan Indonesia untuk **September 2026 ≈ 1.52%**, masih di dalam koridor target Bank Indonesia meski di batas bawah.

> Alur kerja: `data_prep.py` → `model_forecast.py` → `evaluate_model.py` → `optimize_model.py` → `optimize_model_v2.py` → `forecast_final.py`

## 1. Persiapan Data — Menggabungkan 16 Tahun Data BPS

Data mentah berupa 16 file CSV terpisah (satu file per tahun, 2010–2025). Setiap file punya format khas BPS yang perlu dibersihkan:

- **3 baris metadata** di atas (judul, satuan, tahun) sebelum header sebenarnya.
- Nama bulan dalam **Bahasa Indonesia** (`Januari`, `Februari`, …) yang perlu dipetakan ke angka.
- Kolom `Tahunan` yang harus dibuang (bukan data bulanan).
- Nilai kosong ditandai `-` yang harus dikonversi ke `NaN`.

**Langkah pembersihan** (diimplementasikan di `src/data_prep.py`):

1. Baca dan gabungkan seluruh CSV, ambil tahun dari nama file.
2. Petakan nama bulan Indonesia → angka, bentuk `datetime` format `YYYY-MM-01`.
3. Pivot ke **format wide**: index = `datetime` (189 bulan, urut lama→baru), kolom = 151 wilayah.
4. Isi *missing values* (11.808 sel) dengan **interpolasi time-series** (`interpolate(method='time')`).
5. Simpan ke `data/cleaned_inflasi.csv`.

**Dari sudut pandang bisnis:** langkah ini mengubah puluhan lembar laporan tahunan yang tidak konsisten menjadi satu sumber data tunggal yang siap dianalisis — fondasi yang wajib benar sebelum model apa pun dibangun.

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet

warnings.simplefilter("ignore", category=FutureWarning)

# Notebook berjalan dari folder notebooks/, jadi naik satu level ke root proyek
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = BASE_DIR / "data" / "cleaned_inflasi.csv"
TARGET = "INDONESIA"

# Muat data bersih hasil data_prep.py
df = pd.read_csv(DATA_PATH, parse_dates=["datetime"], index_col="datetime").sort_index()
print(f"Dimensi data : {df.shape[0]} bulan x {df.shape[1]} wilayah")
print(f"Rentang      : {df.index.min():%Y-%m} s/d {df.index.max():%Y-%m}")
print(f"Kolom target : {TARGET} (tersedia: {TARGET in df.columns})")
df[[TARGET]].head()

## 2. Forecast Awal — Prophet pada Data M-to-M

Model pertama (`src/model_forecast.py`) menggunakan target apa adanya: **inflasi M-to-M** (perubahan harga bulan ini vs bulan lalu). Prophet mewajibkan dua kolom: `ds` (tanggal) dan `y` (nilai).

Model dilatih dengan `yearly_seasonality=True` untuk menangkap pola musiman tahunan (mis. lonjakan menjelang Lebaran, deflasi pascapanen), lalu memprediksi 12 bulan ke depan.

In [ ]:
def to_prophet_frame(series):
    """Ubah Series ber-index datetime menjadi DataFrame ds/y untuk Prophet."""
    return pd.DataFrame({"ds": series.index, "y": series.values}).dropna()

mtm = df[TARGET].sort_index()
df_mtm = to_prophet_frame(mtm)

model_mtm = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model_mtm.fit(df_mtm)

future_mtm = model_mtm.make_future_dataframe(periods=12, freq="MS")
fc_mtm = model_mtm.predict(future_mtm)

last = fc_mtm.iloc[-1]
print(f"Estimasi M-to-M {last['ds']:%Y-%m}: {last['yhat']:.4f}%")
print(f"  rentang: {last['yhat_lower']:.4f}% s/d {last['yhat_upper']:.4f}%")

Estimasi M-to-M untuk September 2026 keluar di angka **≈ -0.08%** dengan rentang yang **melintasi nol** (-0.62% s/d +0.43%). Prediksi negatif tipis ini masuk akal secara musiman — September memang cenderung deflasi — tetapi ketidakpastiannya besar. Ini yang mendorong kami mengevaluasi akurasi model secara formal.

## 3. Evaluasi Akurasi — Cross-Validation

`src/evaluate_model.py` menjalankan **time-series cross-validation** Prophet dengan skema:

| Parameter | Nilai | Arti bisnis |
|-----------|-------|-------------|
| `initial` | 3650 hari (10 th) | Gunakan 10 tahun pertama sebagai latihan mutlak |
| `period`  | 180 hari (6 bln)  | Ulangi evaluasi, geser jendela tiap 6 bulan |
| `horizon` | 365 hari (12 bln) | Ukur akurasi prediksi hingga 12 bulan ke depan |

Ini mensimulasikan situasi nyata: "seandainya kita hanya punya data sampai titik X, seberapa jauh meleset prediksi 12 bulannya?"

**Hasil baseline M-to-M:** RMSE **0.42%**, MAE **0.32%**.

**Interpretasi:** nilai inflasi M-to-M biasanya bergerak di rentang ±0.3% per bulan. Artinya MAE 0.32% itu **hampir sebesar magnitudo sinyal yang diprediksi** — model berguna untuk melihat *arah dan pola musiman*, tetapi tidak untuk *angka presisi* satu bulan spesifik.

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

CV_INITIAL, CV_PERIOD, CV_HORIZON = "3650 days", "180 days", "365 days"

def run_cv(df_train, **prophet_kwargs):
    """Fit Prophet lalu kembalikan (rmse, mae) rata-rata dari cross-validation."""
    m = Prophet(weekly_seasonality=False, daily_seasonality=False, **prophet_kwargs)
    m.fit(df_train)
    cv = cross_validation(m, initial=CV_INITIAL, period=CV_PERIOD,
                          horizon=CV_HORIZON, disable_tqdm=True)
    pm = performance_metrics(cv)
    return pm["rmse"].mean(), pm["mae"].mean()

rmse_base, mae_base = run_cv(df_mtm, yearly_seasonality=True)
print(f"Baseline M-to-M -> RMSE={rmse_base:.4f}%  MAE={mae_base:.4f}%")

## 4. Optimasi — Dua Percobaan yang Mengajarkan Sesuatu

### 4a. Percobaan pertama yang GAGAL (`optimize_model.py`)

Hipotesis awal: tambahkan efek hari libur nasional (`add_country_holidays('ID')`) dan **naikkan** fleksibilitas tren (`changepoint_prior_scale=0.1`). Hasilnya justru **memperburuk** error (MAE naik +5.26%).

**Pelajaran:** pada data yang sudah bising seperti M-to-M, menambah fleksibilitas membuat model **overfit** noise di data latih lalu meleset saat ekstrapolasi. Efek Lebaran juga bergeser tanggalnya tiap tahun (kalender Hijriah), sehingga regressor libru bertanggal-tetap malah menambah derau. Arah yang benar adalah membuat model **lebih sederhana**, bukan lebih rumit.

### 4b. Grid Search yang benar (`optimize_model_v2.py`)

Kami uji `changepoint_prior_scale ∈ {0.01, 0.02, 0.05}`. Pemenangnya **cps=0.02** (MAE 0.3175%) — menang tipis, mengonfirmasi bahwa menurunkan fleksibilitas memang arah yang tepat, tapi ruang perbaikan M-to-M sudah mentok.

In [ ]:
# Grid search changepoint_prior_scale pada target M-to-M
grid = [0.01, 0.02, 0.05]
results = []
for cps in grid:
    rmse, mae = run_cv(df_mtm, changepoint_prior_scale=cps, yearly_seasonality=True)
    results.append({"cps": cps, "rmse": rmse, "mae": mae})
    print(f"  cps={cps:<5} -> RMSE={rmse:.4f}%  MAE={mae:.4f}%")

best_mtm = min(results, key=lambda r: r["mae"])
print(f"\n-> M-to-M terbaik: cps={best_mtm['cps']} (MAE={best_mtm['mae']:.4f}%)")

## 5. Pivot ke Target Y-o-Y — dan "Jebakan Skala MAE"

### Menghitung Y-o-Y dengan benar

Data kita adalah **laju bulanan (persen)**, bukan Indeks Harga Konsumen (IHK) mentah. Maka Y-o-Y **tidak boleh** dihitung dengan `pct_change(12)` langsung — itu keliru secara matematis. Cara yang benar:

1. **Rekonstruksi indeks harga**: `IHK_t = cumprod(1 + mtm_t/100)`
2. **Hitung Y-o-Y**: `(IHK_t / IHK_{t-12} - 1) * 100`

### Jebakan skala

Model Y-o-Y menghasilkan MAE **1.42%** — jauh lebih besar dari M-to-M (0.32%). Pada pandangan pertama seolah Y-o-Y **lebih buruk**. Ini **salah**, dan inilah inti "jebakan skala":

| Target | Magnitudo sinyal | MAE | **Error relatif** |
|--------|------------------|-----|-------------------|
| M-to-M | ± ~0.3% / bulan | 0.32% | **~100%** (nyaris seburuk menebak nol) |
| Y-o-Y  | ~2–8% / tahun    | 1.42% | **~20–30%** |

**Bahasa bisnis:** membandingkan angka error mentah dari dua skala berbeda itu seperti membandingkan "meleset Rp500" saat menaksir harga permen vs saat menaksir harga mobil. Rp500 fatal untuk permen, tapi tak berarti untuk mobil. Y-o-Y punya MAE absolut lebih besar **hanya karena angkanya sendiri jauh lebih besar** — secara proporsional, prediksinya justru **jauh lebih tepat**.

In [ ]:
def compute_yoy(mtm_series):
    """M-to-M (persen) -> Y-o-Y (persen) via rekonstruksi indeks harga."""
    price_index = (1 + mtm_series / 100.0).cumprod()
    yoy = (price_index / price_index.shift(12) - 1) * 100.0
    return yoy.dropna()  # 12 bulan pertama hilang (tak ada pembanding tahun lalu)

yoy = compute_yoy(mtm)
df_yoy = to_prophet_frame(yoy)

rmse_yoy, mae_yoy = run_cv(df_yoy, yearly_seasonality=True)
print(f"Y-o-Y (default) -> RMSE={rmse_yoy:.4f}%  MAE={mae_yoy:.4f}%")
print(f"Rentang Y-o-Y   : {df_yoy['ds'].min():%Y-%m} s/d {df_yoy['ds'].max():%Y-%m} ({len(df_yoy)} baris)")

In [ ]:
# Perbandingan akhir seluruh pendekatan
print(f"{'Pendekatan':<34}{'RMSE':>10}{'MAE':>10}{'Error relatif':>16}")
print("-" * 70)
print(f"{'Baseline M-to-M (cps=0.05)':<34}{rmse_base:>9.4f}%{mae_base:>9.4f}%{'~100% (buruk)':>16}")
print(f"{'M-to-M terbaik (cps='+str(best_mtm['cps'])+')':<34}{best_mtm['rmse']:>9.4f}%{best_mtm['mae']:>9.4f}%{'~100% (buruk)':>16}")
print(f"{'Y-o-Y (default)':<34}{rmse_yoy:>9.4f}%{mae_yoy:>9.4f}%{'~20-30% (baik)':>16}")

## 6. Forecast Final — Y-o-Y 12 Bulan ke Depan

`src/forecast_final.py` melatih Prophet pada target Y-o-Y dan memprediksi 12 bulan ke depan. Plot menampilkan data historis aktual, garis prediksi, dan pita ketidakpastian (upper/lower). Grafik tersimpan sebagai `forecast_yoy_plot.png`.

In [ ]:
model_yoy = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model_yoy.fit(df_yoy)

future_yoy = model_yoy.make_future_dataframe(periods=12, freq="MS")
fc_yoy = model_yoy.predict(future_yoy)

# --- Visualisasi: historis aktual + prediksi + pita ketidakpastian ---
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df_yoy["ds"], df_yoy["y"], color="#1f77b4", lw=1.6, label="Aktual (historis)")
ax.plot(fc_yoy["ds"], fc_yoy["yhat"], color="#d62728", lw=1.8, label="Prediksi (yhat)")
ax.fill_between(fc_yoy["ds"], fc_yoy["yhat_lower"], fc_yoy["yhat_upper"],
                color="#d62728", alpha=0.15, label="Interval ketidakpastian")
split = df_yoy["ds"].max()
ax.axvline(split, color="gray", ls="--", lw=1, alpha=0.7)
ax.set_title("Inflasi Y-o-Y Indonesia — Historis & Forecast 12 Bulan", fontsize=13)
ax.set_xlabel("Tanggal"); ax.set_ylabel("Inflasi Y-o-Y (%)")
ax.legend(loc="upper right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_DIR / "forecast_yoy_plot.png", dpi=120)
plt.show()

final = fc_yoy.iloc[-1]
print(f"Estimasi inflasi Y-o-Y {final['ds']:%Y-%m}: {final['yhat']:.4f}%")
print(f"  rentang: {final['yhat_lower']:.4f}% s/d {final['yhat_upper']:.4f}%")

## 7. Kesimpulan Akhir

### Rekomendasi: gunakan target yang tepat untuk pertanyaan yang tepat

| Kebutuhan | Target yang dipakai | Alasan |
|-----------|--------------------|--------|
| **Forecasting angka presisi** (mis. proyeksi inflasi tahunan untuk perencanaan) | **Y-o-Y** | Deret halus & stabil; error relatif hanya ~20–30%. Angka yang dihasilkan dapat dipercaya dan dipresentasikan. |
| **Deteksi pola musiman** (kapan spike Lebaran, kapan deflasi panen) | **M-to-M** | Menonjolkan dinamika bulanan yang "tersembunyi" di dalam agregat tahunan. Berguna untuk timing operasional. |

### Poin kunci

1. **Kualitas data adalah fondasi.** Tanpa penggabungan & pembersihan 16 file BPS yang benar (mapping bulan, interpolasi, format wide), model apa pun akan cacat sejak awal.
2. **Model yang lebih rumit ≠ lebih baik.** Menambah holiday dan fleksibilitas tren justru memperburuk hasil pada data bising — menuntun kami menyederhanakan, bukan memperumit.
3. **Jangan tertipu skala MAE.** Angka error harus selalu dibaca relatif terhadap magnitudo sinyalnya. Ini alasan Y-o-Y menang meski MAE absolutnya lebih besar.
4. **Estimasi final:** inflasi Y-o-Y Indonesia **≈ 1.52%** pada September 2026 — di dalam koridor target BI, dengan catatan interval yang masih lebar karena model tidak menyertakan variabel makro eksternal (BBM, kurs, suku bunga).

> **Batasan:** data historis hanya sampai September 2025, dan Prophet murni mengekstrapolasi tren + musiman. Untuk akurasi lebih tinggi, langkah lanjutan adalah menambahkan regressor ekonomi eksternal sebagai variabel bebas.